In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numbers
import pandas as pd
from io import StringIO
import numpy as np
import os


def transform_variable(df: pd.DataFrame, x: str) -> pd.Series:
    if isinstance(df[x].iloc[0], numbers.Number):
        return df[x]
    else:
        return pd.Series([i for i in range(0, len(df[x]))])

def linear_regression(df: pd.DataFrame, x: str, y: str) -> dict:
    fixed_x = transform_variable(df, x)
    model = sm.OLS(list(df[y]), sm.add_constant(fixed_x)).fit()
    bands = pd.read_html(StringIO(model.summary().tables[1].as_html()), header=0, index_col=0)[0]
    coef = pd.read_html(StringIO(model.summary().tables[1].as_html()), header=0, index_col=0)[0]['coef']
    r_2_t = pd.read_html(StringIO(model.summary().tables[0].as_html()), header=None, index_col=None)[0]
    return {
        'm': coef.values[1],
        'b': coef.values[0],
        'r2': r_2_t.values[0][3],
        'r2_adj': r_2_t.values[1][3],
        'low_band': bands['[0.025'].iloc[0], 
        'hi_band': bands['0.975]'].iloc[0]
    }

def plt_lr(df, x, y, m, b, r2, r2_adj, low_band, hi_band, colors):
    fixed_x = transform_variable(df, x)
    plt.plot(df[x], [m * x + b for _, x in fixed_x.items()], color=colors[0])
    plt.fill_between(df[x],
                     [m * x + low_band for _, x in fixed_x.items()],
                     [m * x + hi_band for _, x in fixed_x.items()],
                     alpha=0.2, color=colors[1])

path = "img/Semana 13"

df = pd.read_csv("csv/aprovCreditos.csv", index_col=False)
os.makedirs(path, exist_ok=True)


a=linear_regression(df, "Approved_Declined_Amount", "Disbursed_Shipped_Amount")

plt_lr(df=df, x="Approved_Declined_Amount", y="Disbursed_Shipped_Amount", colors=('red', 'orange'), **a)
plt.xticks(rotation=90)
plt.savefig(f'{path}/lr_App_Dec_Disbursed.png')
plt.close()

print(f"R2: {a["r2"]}")
print(f"R2 ajustado: {a["r2_adj"]}")

df_by_year = df.groupby("Decision_Year").agg(
    total=pd.NamedAgg(column="Approved_Declined_Amount", aggfunc="sum")
).reset_index()

a=linear_regression(df_by_year, "Decision_Year", "total")
plt_lr(df=df, x="Approved_Declined_Amount", y="Disbursed_Shipped_Amount", colors=('red', 'orange'), **a)
plt.xticks(rotation=90)
plt.savefig(f'{path}/lr_Decision_Year_total.png')
plt.close()

print(f"R2: {a["r2"]}")
print(f"R2 ajustado: {a["r2_adj"]}")

R2: 0.65
R2 ajustado: 0.65
R2: 0.234
R2 ajustado: 0.192
